In [29]:
import pandas as pd

In [30]:
folder_raw = '/home/gflameida2/repositorios/pipeline-commerce/data/raw/'
folder_interim = '/home/gflameida2/repositorios/pipeline-commerce/data/interim/'
folder_logs = '/home/gflameida2/repositorios/pipeline-commerce/data/logs/'
categorias_produto = pd.read_csv(folder_raw + 'categorias_produto.csv', sep=';')
clientes = pd.read_csv(folder_raw + 'clientes.csv', sep=';')
entregas = pd.read_csv(folder_raw + 'entregas.csv', sep=';')
estoque_movimentacoes = pd.read_csv(folder_raw + 'estoque_movimentacoes.csv', sep=';')
itens_pedido = pd.read_csv(folder_raw + 'itens_pedido.csv', sep=';')
lojas = pd.read_csv(folder_raw + 'lojas.csv', sep=';')
pagamentos = pd.read_csv(folder_raw + 'pagamentos.csv', sep=';')
pedidos = pd.read_csv(folder_raw + 'pedidos.csv', sep=';')
produtos = pd.read_csv(folder_raw + 'produtos.csv', sep=';')
vendedores = pd.read_csv(folder_raw + 'vendedores.csv', sep=';')


In [31]:
lista_df = [categorias_produto, clientes, entregas, estoque_movimentacoes, itens_pedido, lojas, pagamentos, pedidos, produtos, vendedores]
lista_df_names = ['categorias_produto', 'clientes', 'entregas', 'estoque_movimentacoes', 'itens_pedido', 'lojas', 'pagamentos', 'pedidos', 'produtos', 'vendedores']
primary_key = ['categoria_id', 'cliente_id', 'entrega_id', 'movimento_id', 'item_pedido_id', 'loja_id', 'pagamento_id', 'pedido_id', 'produto_id', 'vendedor_id']

In [36]:
invalid_emails = clientes[~clientes['email'].str.contains('@', na=False)]
print(invalid_emails)

      cliente_id tipo_cliente    nome_razao_social           documento  \
167          168           PF      William Almeida      788.348.003-51   
370          371           PJ  Prime Distribuidora      409.869.011-93   
603          604           PJ       Nova Papelaria  02.881.103/4281-76   
694          695           PF       Igor Fernandes      864.424.617-54   
875          876           PF        Kleber Santos      023.271.919-44   
1500        1501           PF     Juliana Oliveira      678.871.468-09   
1501        1502           PF         Kleber Costa      626.398.833-78   
1502        1503           PF       Kleber Pereira      735.141.406-17   
1503        1504           PJ           Rio Varejo      464.585.448-81   
1504        1505           PF      Otávio Teixeira      952.536.595-78   
1505        1506           PF       Rafael Freitas      473.405.759-28   
1506        1507           PF        Natália Moura      941.461.428-42   
1507        1508           PF     Carl

# FIXING ERROS

In [27]:
itens_pedido['desconto_item'] = itens_pedido['desconto_item'].str.replace(',', '.').astype(float)
itens_pedido['preco_unitario'] = itens_pedido['preco_unitario'].astype(float)

clientes['data_nascimento'] = pd.to_datetime(clientes['data_nascimento'], format='%d/%m/%Y')
clientes['data_cadastro'] = pd.to_datetime(clientes['data_cadastro'], errors='coerce')


# Handling formart issues in 'pedidos' DataFrame
pedidos['valor_frete'] = pedidos['valor_frete'].str.replace(',', '.').astype(float)
pedidos['data_pedido'] = pedidos['data_pedido'].apply(lambda x: x + ':00' if len(x.split(':')) == 2 else x)
pedidos['data_pedido'] = pedidos['data_pedido'].str.replace('/', '-', regex=False)
pedidos['data_pedido'] = pd.to_datetime(pedidos['data_pedido'], dayfirst=True, errors='coerce')


estoque_movimentacoes['quantidade_movimentada'] = estoque_movimentacoes['quantidade_movimentada'].replace('dez','10')
estoque_movimentacoes['quantidade_movimentada'] = estoque_movimentacoes['quantidade_movimentada'].astype(int)
estoque_movimentacoes['data_movimento'] = estoque_movimentacoes['data_movimento'].apply(lambda x: x + ':00' if len(x.split(':')) == 2 else x)
estoque_movimentacoes['data_movimento'] = estoque_movimentacoes['data_movimento'].str.replace('/', '-', regex=False)
estoque_movimentacoes['data_movimento'] = pd.to_datetime(estoque_movimentacoes['data_movimento'], dayfirst=True, errors='coerce')

entregas['data_prevista'] = pd.to_datetime(entregas['data_prevista'], errors='coerce')

produtos['preco_unitario'] = produtos['preco_unitario'].str.replace('R$ ', '').replace('.', '')
produtos['preco_unitario'] = produtos['preco_unitario'].str.replace(',', '.').astype(float)

/tmp/ipykernel_1917987/3392025093.py:12: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  pedidos['data_pedido'] = pd.to_datetime(pedidos['data_pedido'], dayfirst=True, errors='coerce')
/tmp/ipykernel_1917987/3392025093.py:19: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  estoque_movimentacoes['data_movimento'] = pd.to_datetime(estoque_movimentacoes['data_movimento'], dayfirst=True, errors='coerce')


In [28]:

for df, name in zip(lista_df, lista_df_names):
    print('---' * 20)
    print(f"DataFrame: {name}")
    
    # Identify rows with null values
    null_rows = df[df.isnull().any(axis=1)]
    if not null_rows.empty:
        null_rows.to_csv(folder_logs + f'{name}_null.csv', index=False, sep=';')
        
    # Drop rows with null values from the original DataFrame
    df_cleaned = df.dropna()
    
    print('Nulos after cleaning: ')
    print(df_cleaned.isnull().sum())
    
    df_cleaned.to_csv(folder_interim + f'{name}.csv', index=False, sep=';')


------------------------------------------------------------
DataFrame: categorias_produto
Nulos after cleaning: 
categoria_id        0
nome_categoria      0
status_categoria    0
dtype: int64
------------------------------------------------------------
DataFrame: clientes
Nulos after cleaning: 
cliente_id           0
tipo_cliente         0
nome_razao_social    0
documento            0
email                0
telefone             0
cidade               0
uf                   0
data_cadastro        0
data_nascimento      0
status_cliente       0
dtype: int64
------------------------------------------------------------
DataFrame: entregas
Nulos after cleaning: 
entrega_id           0
pedido_id            0
transportadora       0
data_postagem        0
data_prevista        0
data_entrega_real    0
status_entrega       0
modalidade_frete     0
dtype: int64
------------------------------------------------------------
DataFrame: estoque_movimentacoes
Nulos after cleaning: 
movimento_id       

In [ ]:
folder_interim = '/home/gflameida2/repositorios/pipeline-commerce/data/interim/'
categorias_produto_interim = pd.read_csv(folder_interim + 'categorias_produto.csv', sep=';')
clientes_interim = pd.read_csv(folder_interim + 'clientes.csv', sep=';')
entregas_interim = pd.read_csv(folder_interim + 'entregas.csv', sep=';')
estoque_movimentacoes_interim = pd.read_csv(folder_interim + 'estoque_movimentacoes.csv', sep=';')
itens_pedido_interim = pd.read_csv(folder_interim + 'itens_pedido.csv', sep=';')
lojas_interim = pd.read_csv(folder_interim + 'lojas.csv', sep=';')
pagamentos_interim = pd.read_csv(folder_interim + 'pagamentos.csv', sep=';')
pedidos_interim = pd.read_csv(folder_interim + 'pedidos.csv', sep=';')
produtos_interim = pd.read_csv(folder_interim + 'produtos.csv', sep=';')
vendedores_interim = pd.read_csv(folder_interim + 'vendedores.csv', sep=';')


In [ ]:
estoque_movimentacoes_interim['quantidade_movimentada'][5600]